# Week 4 — Fine-Tune TrOCR on the Urdu OCR Dataset (Colab version)

This picks up where Week 3 left off: `labels.csv` and the generated images from Week 3 get fine-tuned into `microsoft/trocr-base-printed` (Microsoft's pretrained OCR model), then evaluated on a held-out test set.

**This version runs on Google Colab, not the Codespace.** Colab gives you a free GPU, but its filesystem is wiped every time the runtime disconnects — so the workflow here is:

1. **Step 0** — clone your GitHub repo into the Colab runtime to get Week 3's `labels.csv` + images.
2. **Steps 1–4** — same training/evaluation as the handout (unchanged from the Codespace version).
3. **Step 5** — instead of just saving the model to local disk (which disappears when the runtime dies), push the trained model to the **Hugging Face Hub** so it survives past this session and Week 5 can reload it with one line.
4. **Saving the notebook itself** — don't push it from a code cell. Use Colab's built-in **File → Save a copy in GitHub** button once you're done. That commits only the `.ipynb` file back to your repo; it never touches the cloned data or the trained model weights (those live in the Hub repo instead, which is where multi-GB model files belong anyway — git repos handle large binary files badly).

**A note on `TensorFlow_with_GPU.ipynb`:** that notebook's GPU-connectivity check is TensorFlow-specific (`tf.test.gpu_device_name()`). This project's stack is PyTorch + Hugging Face Transformers, so the equivalent check below uses `torch.cuda.is_available()` instead.

**What follows the official handout, and what's added on top:**
- Steps 1–4 below match the Week 4 handout's structure and step numbers.
- **Step 0** is new — a fresh Colab runtime starts empty, so it has to exist to pull in Week 3's data.
- **Step 5** is new — pushing to the Hugging Face Hub instead of only saving to local disk, which is required to persist anything past the Colab session.
- Sections marked **Extra** are additions beyond the handout: a training-loss chart, Character/Word Error Rate alongside exact-match accuracy, automatic extraction of the worst predictions, a visual grid of sample predictions, and per-epoch checkpointing to disk with automatic resume.
- Three real bugs get fixed along the way (verified in a sandboxed test against `transformers==4.57.6`, the version pinned in Week 3): `from transformers import AdamW` no longer exists in this version and raises `ImportError` — `torch.optim.AdamW` is the fix; the labels Week 3 produced pad with the literal pad-token id instead of `-100`, so the loss function was never actually ignoring padding — fixed in the Dataset class below; and the handout's model name has a dropped hyphen (`trocr-baseprinted`) — corrected to `trocr-base-printed` to match Week 3 and the handout's own Sources section.


## Audit notes (this pass)

This version was put through a performance/accuracy audit. Two findings drove almost
everything below:

**A fourth bug — bigger than the three above — was silently dropping 58% of the dataset.**
`labels.csv` mixes two path conventions: Week 1's `books`/`newspaper`/`other` rows (153 of
263) store image paths like `data/raw/other/utrset_003.png`, relative to `SI26-Week1/`
itself, while Week 3's `synthetic` rows (110 of 263) store paths like
`raw/synthetic/urdu_12.png`, relative to `SI26-Week1/data/`. `os.path.join(DATA_DIR, ...)`
can only ever satisfy one of those conventions, so every Week-1 row was quietly treated as
"missing" and skipped — even though the files were sitting right there on disk the whole
time. Week 3's own diagnostics flagged "153 missing files" and blamed a failed UTRSet zip
download; that diagnosis was wrong; it was a path bug, not a missing download. Fixed below
with a fallback path resolver (`resolve_image_path`) that recovers all 263 rows — see
Step 2.

**With that bug in place, the model's actual test accuracy was 0% (0/22) — every single
prediction decoded to the Unicode replacement character (`�`), and CER was 2.51 (worse than
guessing nothing).** That's not really a hyperparameter problem, it's a "the model never
got a real chance" problem: only 88 of 263 lines ever reached training, across 3 epochs —
66 gradient steps total — of a 334M-parameter model whose text decoder (RoBERTa) was
pretrained only on English and has never seen Urdu script (see the note in Step 1). The
changes below — recovered data, a right-sized `MAX_LENGTH`, ~20x more training steps,
mixed precision, LR warmup, gradient clipping, and per-epoch validation with best-checkpoint
selection — give the model a realistic shot. They're very likely to move accuracy off of
0%; whether they get it to a genuinely *good* number is a more open question given the
decoder's English-only pretraining, and is worth watching for honestly rather than assuming
away (more on this in Step 1 and Step 2 Extra: Right-Size `MAX_LENGTH`).

## Step 0: Clone Your Repo to Get Week 3's Data

*(Not in the official handout.)* A fresh Colab runtime starts completely empty — there's no Codespace disk to fall back on. This cell clones your GitHub repo (the same one Week 1–3 used) so `labels.csv` and the generated images exist locally in the Colab VM.

**Fill in `REPO_URL` below with your repo's clone URL** (e.g. `https://github.com/<your-username>/Urdu-OCR-Project-Code-Saviours-SI-26-Humna-Imran.git`). If the repo is private, use a URL with a token: `https://<token>@github.com/<user>/<repo>.git`, or set up SSH — don't hardcode a personal GitHub token in a shared/submitted notebook.

In [ ]:
import os
import pandas as pd

# --- Fill this in with your repo's clone URL ---
REPO_URL = "https://github.com/hamnasz/Urdu-OCR-Project-Code-Saviours-SI-26-Humna-Imran.git"
REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

if not os.path.isdir(REPO_DIR):
    !git clone "{REPO_URL}" "{REPO_DIR}"
else:
    print(f"{REPO_DIR} already present in this runtime, skipping clone.")

# Same relative layout as the Codespace version, just rooted at the freshly cloned copy.
DATA_DIR = os.path.join(REPO_DIR, "SI26-Week1", "data")

LABELS_PATH = os.path.join(DATA_DIR, "labels.csv")
print(f"Found labels.csv with {len(pd.read_csv(LABELS_PATH))} rows at {LABELS_PATH}")

### Step 0b: Connect to the Hugging Face Hub

*(Not in the handout.)* The trained model gets pushed to the Hub in Step 5, so authenticate now.

**Don't paste your HF token directly into a cell** — anyone you share the notebook with (or a public GitHub push) would see it in the source. Instead, use Colab's built-in **Secrets** manager:

1. Click the 🔑 key icon in the left sidebar of Colab.
2. Add a new secret named `HF_TOKEN`, paste your Hugging Face **write** token as the value (get one at huggingface.co → Settings → Access Tokens), and toggle "Notebook access" on.
3. Run the cell below — it reads the secret at runtime, so the token itself never appears in the notebook file or its output.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")  # never print this variable
login(token=HF_TOKEN)
print("Logged in to Hugging Face Hub.")

## Step 1: Load the Pretrained TrOCR Model

TrOCR combines a vision encoder (reads the image) with a text decoder (outputs characters). This loads the version already trained on printed text, before fine-tuning it on the Urdu images from Week 3 — transfer learning.

**Worth knowing before fine-tuning starts:** this model's text decoder is RoBERTa,
pretrained on English text — it has never seen Urdu script. The vision encoder's job
(reading strokes/shapes from pixels) transfers across scripts reasonably well, but the
decoder has to learn Urdu's token vocabulary essentially from scratch during fine-tuning.
That's a big part of why this notebook needs far more than the handout's 3 epochs to
produce non-garbage output — see the intro cell above and Step 2 Extra: Right-Size
`MAX_LENGTH` below.

In [ ]:
# NOTE: torch is intentionally left out of this upgrade. Colab's preinstalled torch
# is already matched to its preinstalled torchvision -- upgrading torch alone (leaving
# torchvision behind) breaks torchvision's compiled ops and causes:
#   RuntimeError: operator torchvision::nms does not exist
# when transformers tries to import it. If you already hit that error in this session,
# fix isn't enough on its own -- go to Runtime > Restart session first, then re-run all
# cells from the top.
# Pillow 12.0.0 shipped with a real bug: PIL/ImageText.py imports a type called _Ink
# from PIL/_typing.py, but that release never actually defined it there. Any package
# that touches PIL.ImageDraw/ImageFont (torchvision does, indirectly, when transformers
# imports it) then fails with:
#   ImportError: cannot import name '_Ink' from 'PIL._typing'
# Pinning below version 12 avoids the broken release entirely.
!pip install --upgrade "transformers==4.57.6" "pillow<12" pandas sentencepiece protobuf jiwer matplotlib huggingface_hub --quiet

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU.")
    print("Training will still run on CPU, just far slower (hours instead of minutes).")
else:
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

# Handout text has this as 'microsoft/trocr-baseprinted' (missing hyphen) -- that's not
# a real model on the Hub. The correct id (matching Week 3 and the handout's own Sources
# section) is:
MODEL_NAME = "microsoft/trocr-base-printed"

try:
    processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
    model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)
except Exception as e:
    raise RuntimeError(
        f"Couldn't load '{MODEL_NAME}' from Hugging Face Hub: {e}. "
        "Check your internet connection and that huggingface.co is reachable."
    ) from e

model = model.to(device)

# Configure model for generation (standard TrOCR fine-tuning setup)
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print("Model loaded successfully!")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## Step 2: Set Up Training

A `DataLoader` wraps the dataset and feeds it to the model in small batches — batch size 8 means the model sees 8 images at a time before updating its weights. The optimiser (`AdamW`) controls how the model adjusts itself after each batch.

The `UrduOCRDataset` class from Week 3 is redefined here so this notebook can run standalone in a fresh Colab session — skip this cell if Week 3's version is already in memory. One change from Week 3: padded label positions are now set to `-100` instead of the pad-token id. PyTorch's `CrossEntropyLoss` (which the model uses internally) ignores `-100` by default but nothing else — so Week 3's version was training the model to predict padding too, which quietly hurts both the loss numbers and generation quality. This is the fix the official TrOCR fine-tuning guide uses.

**Audit-pass additions to this cell:** `UrduOCRDataset` now resolves each row's image path
two ways (see `resolve_image_path` below) instead of one, which recovers 153 rows that the
original version silently dropped — see the intro cell for why. It also now precomputes
every `pixel_values`/`labels` tensor once in `__init__` instead of redoing PIL decode +
resize + tokenize on every single `__getitem__` call — with training now running for many
more epochs, repeating that work every epoch for images that never change was pure waste.
263 cached image tensors is well under 500MB, so this comfortably fits in memory. Finally,
the train/test split below is now stratified by category (and adds a validation slice) —
see the comment in the split code for why.

### Step 2 Extra: Right-Size `MAX_LENGTH`

*(Not in the handout.)* The handout hardcodes `MAX_LENGTH = 128` as a guess. This measures
it instead: every label in `labels.csv` gets tokenized with the real TrOCR tokenizer, and
`MAX_LENGTH` is set to the longest sequence actually found (plus a small buffer for the
special tokens `generate()` adds), never below the handout's original 128.

This matters more here than it might elsewhere: Urdu isn't in this tokenizer's training
data (previous cell), so it falls back to encoding raw UTF-8 bytes rather than efficient
learned subwords, and Urdu characters take ~1.8 UTF-8 bytes each on average versus close to
1 for English. That inflates token counts well past what 128 was likely sized for, which
means labels were silently getting truncated during training on top of everything else.

In [ ]:
import pandas as pd

_texts = pd.read_csv(LABELS_PATH)["text"].astype(str).tolist()
_lengths = [len(processor.tokenizer(t).input_ids) for t in _texts]

print(f"Tokenized length over {len(_texts)} labels -- "
      f"min: {min(_lengths)}, mean: {sum(_lengths) / len(_lengths):.1f}, max: {max(_lengths)}")
print(f"Rows that would truncate at the handout's MAX_LENGTH=128: {sum(l > 128 for l in _lengths)}")

# Longest real sequence + a small buffer for decoder_start/eos, never below the original 128.
MAX_LENGTH = max(128, max(_lengths) + 8)
print(f"Using MAX_LENGTH = {MAX_LENGTH}")

In [ ]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader


def resolve_image_path(data_dir, rel_path):
    """labels.csv mixes two path conventions (Week 1 vs. Week 3 -- see the intro cell).
    Try the direct join first (covers Week 3's ~110 'synthetic' rows, stored relative to
    DATA_DIR), then fall back to stripping a redundant leading 'data/' (covers Week 1's
    ~153 'books'/'newspaper'/'other' rows, stored relative to DATA_DIR's parent). Recovers
    all 263 rows instead of 110. Reused in Step 3 Extra's prediction grid, which hits the
    exact same path-joining logic when it maps results back to image files.
    """
    direct = os.path.join(data_dir, rel_path)
    if os.path.isfile(direct):
        return direct
    if rel_path.startswith("data/"):
        stripped = os.path.join(data_dir, rel_path[len("data/"):])
        if os.path.isfile(stripped):
            return stripped
    return direct  # neither resolves -- caller's exists-check reports this one as genuinely missing


class UrduOCRDataset(Dataset):
    """PyTorch Dataset over labels.csv, skipping rows with no image on disk.

    Each item returns TrOCR-ready pixel_values and tokenized labels, with padding
    positions set to -100 so the loss function ignores them (the Week-3-vs-official-guide
    fix noted above). pixel_values/labels are computed once here in __init__ rather than
    on every __getitem__ call -- see the note in the Step 2 intro above for why.
    """

    def __init__(self, csv_path, processor, data_dir, max_length=MAX_LENGTH):
        data = pd.read_csv(csv_path)
        resolved = data["image"].apply(lambda p: resolve_image_path(data_dir, p))
        has_file = resolved.apply(os.path.isfile)
        n_missing = int((~has_file).sum())
        if n_missing:
            print(f"Skipping {n_missing} rows with no image on disk (checked both path conventions)")
        self.data = data[has_file].reset_index(drop=True)
        self.max_length = max_length
        resolved_paths = resolved[has_file].reset_index(drop=True)
        print(f"Dataset loaded: {len(self.data)} samples")

        pad_id = processor.tokenizer.pad_token_id
        self._pixel_values, self._labels = [], []
        for i in range(len(self.data)):
            image = Image.open(resolved_paths.iloc[i]).convert("RGB")
            pv = processor(image, return_tensors="pt").pixel_values.squeeze()
            # Padding/truncating by hand (slice + pad with pad_token_id) instead of passing
            # padding="max_length", truncation=True straight to the tokenizer: verified in a
            # sandbox that some tokenizers/transformers version pairs raise
            # `TypeError: enable_truncation() got an unexpected keyword argument 'direction'`
            # on that call path. Slicing a plain Python list never depends on that internal API.
            raw_ids = processor.tokenizer(str(self.data.iloc[i]["text"])).input_ids[: self.max_length]
            ids = raw_ids + [pad_id] * (self.max_length - len(raw_ids))
            ids = [t if t != pad_id else -100 for t in ids]
            self._pixel_values.append(pv)
            self._labels.append(torch.tensor(ids))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return {"pixel_values": self._pixel_values[idx], "labels": self._labels[idx]}


dataset = UrduOCRDataset(LABELS_PATH, processor, data_dir=DATA_DIR)

# Stratified split by category (roughly 70/15/15 train/val/test) instead of the handout's
# plain 80/20 random split. Two changes from the original:
#   1. A plain random split doesn't guarantee train and test see the same mix of
#      books/newspaper/other/synthetic -- with the path-bug fix now bringing the three
#      real-world categories back into play alongside synthetic, that mix matters more
#      than it did when only synthetic images were reachable.
#   2. There's now a validation slice, separate from the test set, used only to monitor
#      training and pick the best checkpoint (Step 2 Extra: Checkpoint to Disk, below).
#      The test set stays untouched until Step 3.
# Fixed seed for the same reason as before: an identical split across sessions is required
# for checkpoint-resume to never leak val/test images into training.
torch.manual_seed(42)
categories = dataset.data["category"].fillna("uncategorized")
train_indices, val_indices, test_indices = [], [], []
for cat in categories.unique():
    idx = categories.index[categories == cat].to_numpy()
    idx = idx[torch.randperm(len(idx)).numpy()]
    n_test = max(1, int(round(0.15 * len(idx))))
    n_val = max(1, int(round(0.15 * len(idx))))
    test_indices.extend(idx[:n_test].tolist())
    val_indices.extend(idx[n_test:n_test + n_val].tolist())
    train_indices.extend(idx[n_test + n_val:].tolist())

train_dataset = torch.utils.data.Subset(dataset, train_indices)
val_dataset = torch.utils.data.Subset(dataset, val_indices)
test_dataset = torch.utils.data.Subset(dataset, test_indices)
print(f"Train: {len(train_dataset)}  Val: {len(val_dataset)}  Test: {len(test_dataset)}")
print("Train category mix:", categories.loc[train_indices].value_counts().to_dict())
print("Test category mix:", categories.loc[test_indices].value_counts().to_dict())

### Step 2 Extra: Checkpoint to Disk

*(Not in the handout.)* A Codespace can still disconnect mid-training -- an idle timeout, a network hiccup, an accidentally closed browser tab -- which normally means starting the full run over from epoch 0. The cell below checks the `checkpoints/` folder next to your data for a checkpoint from a previous run and, if one exists, resumes from the next epoch instead of the beginning. The training loop after it saves a checkpoint (model weights, optimizer state, and loss history) to disk at the end of every epoch, overwriting the previous one so it doesn't pile up. This only stays correct because of the fixed seed added above -- it keeps the train/test split identical across sessions, so resuming never leaks old training images into the test set.

**Two additions from the audit pass:** (1) the saved checkpoint now also records the
dataset/train size it was written against, and refuses to silently resume if those don't
match what's loaded right now -- this project's very first `checkpoints/` folder was
written by the *pre-fix* pipeline (110 samples, not 263), and blindly resuming from it is
exactly what made the original run report "already trained" without training at all. If you
see the mismatch warning below, delete the old `checkpoints/` folder. (2) every
`EVAL_EVERY` epochs the loop now measures CER on the validation slice (not the test set)
and keeps a separate `best_model/` checkpoint -- with `NUM_EPOCHS` raised well past 3,
there's no guarantee the *last* epoch is the *best* one, and the original notebook had no
way to tell.

In [ ]:
# NOTE: the handout says `from transformers import AdamW` -- that raises ImportError on
# transformers==4.57.6 (verified in a sandbox). Transformers dropped its own AdamW in favor
# of PyTorch's; torch.optim.AdamW is the standard drop-in replacement.
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import json

# --- Hyperparameters (revised during the audit pass -- see the intro cell for the numbers
#     behind these choices) ---
BATCH_SIZE = 8        # was 4. TrOCR-base at 384x384 comfortably fits a bigger batch on a
                       # T4's 16GB, especially with mixed precision below; fewer, bigger
                       # batches also means less DataLoader/Python-loop overhead per epoch.
                       # Lower this back to 4 if the OOM handler below tells you to.
LEARNING_RATE = 5e-5   # unchanged -- a standard fine-tuning LR for this model size. The
                       # original problem wasn't the LR value, it was too few total steps
                       # (see NUM_EPOCHS) and no warmup (see the scheduler below).
NUM_EPOCHS = 20        # was 3. With the path-resolution fix, training now sees ~185 lines/
                       # epoch instead of 88 -- and this notebook now completes roughly 15-20x
                       # more optimizer steps overall. 3 epochs at batch 4 was 66 gradient
                       # steps total for a 334M-parameter model whose decoder has never seen
                       # Urdu script; that's not a hyperparameter problem, it's not enough
                       # training, period. This will run for a good while longer than the
                       # original "20-40 minutes" estimate -- if Colab's session limit is a
                       # problem, lower this and rely on the resume-from-checkpoint logic
                       # below to continue across sessions.
EVAL_EVERY = 2         # compute validation CER (and possibly update best_model/) every N
                       # epochs. Generation is much slower than a forward pass, so doing
                       # this every single epoch would meaningfully slow the loop down.
USE_AMP = device == "cuda"  # mixed precision: real speedup + lower memory on a T4, no-op on CPU.

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=(device == "cuda"))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                         num_workers=2, pin_memory=(device == "cuda"))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                          num_workers=2, pin_memory=(device == "cuda"))

CHECKPOINT_DIR = os.path.join(DATA_DIR, "checkpoints")
BEST_MODEL_DIR = os.path.join(DATA_DIR, "best_model")
CHECKPOINT_STATE_PATH = os.path.join(CHECKPOINT_DIR, "checkpoint_state.json")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

total_steps = len(train_loader) * NUM_EPOCHS


def fresh_optimizer_and_scheduler(m):
    opt = AdamW(m.parameters(), lr=LEARNING_RATE)
    sched = get_linear_schedule_with_warmup(
        opt, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
    )
    return opt, sched


resume_ok = False
if os.path.isfile(CHECKPOINT_STATE_PATH):
    with open(CHECKPOINT_STATE_PATH) as f:
        ckpt_state = json.load(f)
    # Guard against silently resuming from a checkpoint written against a different dataset
    # -- see the note in the markdown cell above.
    if ckpt_state.get("dataset_size") != len(dataset) or ckpt_state.get("train_size") != len(train_dataset):
        print(f"Found a checkpoint at {CHECKPOINT_DIR}, but it doesn't match the current "
              f"dataset (saved dataset_size={ckpt_state.get('dataset_size')}, "
              f"train_size={ckpt_state.get('train_size')} vs. current {len(dataset)}, "
              f"{len(train_dataset)}). Treating it as stale and starting fresh -- delete "
              "the checkpoints/ folder to silence this check.")
    else:
        resume_ok = True

if resume_ok:
    start_epoch = ckpt_state["last_completed_epoch"] + 1
    loss_history = ckpt_state["loss_history"]
    epoch_avg_losses = ckpt_state["epoch_avg_losses"]
    best_val_cer = ckpt_state.get("best_val_cer", float("inf"))
    print(f"Found a valid checkpoint through epoch {start_epoch} at {CHECKPOINT_DIR} -- reloading model weights.")
    model = VisionEncoderDecoderModel.from_pretrained(CHECKPOINT_DIR).to(device)
    optimizer, scheduler = fresh_optimizer_and_scheduler(model)
    optimizer.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, "optimizer.pt"), map_location=device))
    if os.path.isfile(os.path.join(CHECKPOINT_DIR, "scheduler.pt")):
        scheduler.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, "scheduler.pt"), map_location=device))
else:
    print("No valid checkpoint found -- starting training from scratch.")
    optimizer, scheduler = fresh_optimizer_and_scheduler(model)
    start_epoch = 0
    loss_history = []
    epoch_avg_losses = []
    best_val_cer = float("inf")

print(f"Training batches per epoch: {len(train_loader)}  (total planned steps: {total_steps})")
if start_epoch >= NUM_EPOCHS:
    print(f"All {NUM_EPOCHS} epochs already completed per the saved checkpoint -- nothing left to train.")
else:
    print(f"Ready to train epochs {start_epoch + 1} through {NUM_EPOCHS}.")

In [ ]:
# This cell will take noticeably longer than the original 20-40 minute estimate -- many
# more epochs, plus a validation pass every EVAL_EVERY epochs. Safe to re-run after a
# disconnect: it continues from the last saved checkpoint instead of retraining from epoch
# 0 (see the resume/staleness check above).

import jiwer  # also used again in Step 3

scaler = torch.amp.GradScaler(device="cuda", enabled=USE_AMP)


def run_generation_eval(loader, m):
    """Beam-search generation + CER over a loader. Used for the periodic validation
    check here, and reused as-is for the final Step 3 evaluation."""
    m.eval()
    refs, hyps = [], []
    with torch.no_grad():
        for batch in loader:
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].clone()
            labels[labels == -100] = processor.tokenizer.pad_token_id
            generated_ids = m.generate(pixel_values, max_length=MAX_LENGTH, num_beams=4)
            hyps.extend(processor.batch_decode(generated_ids, skip_special_tokens=True))
            refs.extend(processor.batch_decode(labels, skip_special_tokens=True))
    m.train()
    return jiwer.cer(refs, hyps) if refs else float("nan")


for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    total_loss = 0
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    print("-" * 30)
    for batch_idx, batch in enumerate(train_loader):
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        try:
            with torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                outputs = model(pixel_values=pixel_values, labels=labels)
                loss = outputs.loss

            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            # Gradient clipping: standard stabilizer for full-parameter fine-tuning of a
            # large pretrained model on data far outside its original training distribution.
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
        except torch.cuda.OutOfMemoryError:
            print(f"  Batch {batch_idx}: ran out of GPU memory. Lower BATCH_SIZE in the "
                  "cell above and re-run from there.")
            raise

        total_loss += loss.item()
        loss_history.append(loss.item())
        if batch_idx % 10 == 0:
            print(f"  Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f} | "
                  f"LR: {scheduler.get_last_lr()[0]:.2e}")

    avg_loss = total_loss / len(train_loader)
    epoch_avg_losses.append(avg_loss)
    print(f"Epoch {epoch + 1} complete | Average Loss: {avg_loss:.4f}")

    # Checkpoint to disk (Not in the handout) -- overwrites the previous epoch's save.
    model.save_pretrained(CHECKPOINT_DIR)
    processor.save_pretrained(CHECKPOINT_DIR)
    torch.save(optimizer.state_dict(), os.path.join(CHECKPOINT_DIR, "optimizer.pt"))
    torch.save(scheduler.state_dict(), os.path.join(CHECKPOINT_DIR, "scheduler.pt"))

    # Extra: validate every EVAL_EVERY epochs and keep a separate best_model/ checkpoint --
    # NUM_EPOCHS is now high enough that the last epoch isn't guaranteed to be the best one.
    if (epoch + 1) % EVAL_EVERY == 0 or (epoch + 1) == NUM_EPOCHS:
        val_cer = run_generation_eval(val_loader, model)
        print(f"Validation CER after epoch {epoch + 1}: {val_cer:.3f} (best so far: {best_val_cer:.3f})")
        if val_cer < best_val_cer:
            best_val_cer = val_cer
            model.save_pretrained(BEST_MODEL_DIR)
            processor.save_pretrained(BEST_MODEL_DIR)
            print(f"  New best -- saved to {BEST_MODEL_DIR}")

    with open(CHECKPOINT_STATE_PATH, "w") as f:
        json.dump({
            "last_completed_epoch": epoch,
            "loss_history": loss_history,
            "epoch_avg_losses": epoch_avg_losses,
            "best_val_cer": best_val_cer,
            "dataset_size": len(dataset),
            "train_size": len(train_dataset),
        }, f)
    print(f"Checkpoint saved to disk ({CHECKPOINT_DIR})")

print("\nTraining complete!")
print(f"Training loss went from {loss_history[0]:.4f} (first batch) to {loss_history[-1]:.4f} (last batch)")
if best_val_cer < float("inf"):
    print(f"Best validation CER seen during training: {best_val_cer:.3f} (checkpoint saved to {BEST_MODEL_DIR})")

### Step 2 Extra: Visualize Training Progress

*(Not in the handout.)* The submission asks for a screenshot of the training output showing loss decreasing — an actual chart makes that point far more clearly than a screenshot of scrolling console text, and it's this cell's output that best supports the required `'Training loss went from X to X'` line.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(loss_history, color="#4C72B0", linewidth=1.2)
batches_per_epoch = len(train_loader)
for e in range(1, NUM_EPOCHS):
    axes[0].axvline(e * batches_per_epoch, color="#999999", linestyle=":", linewidth=1)
axes[0].set_title("Training Loss per Batch")
axes[0].set_xlabel("Batch (cumulative across epochs)")
axes[0].set_ylabel("Loss")

axes[1].plot(range(1, NUM_EPOCHS + 1), epoch_avg_losses, marker="o", color="#C44E52")
axes[1].set_title("Average Loss per Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Average Loss")
axes[1].set_xticks(range(1, NUM_EPOCHS + 1))
for i, v in enumerate(epoch_avg_losses):
    axes[1].annotate(f"{v:.3f}", (i + 1, v), textcoords="offset points", xytext=(0, 8), ha="center")

plt.tight_layout()
plt.savefig("training_loss.png", dpi=150)
plt.show()
print("Saved to training_loss.png -- attach this for the 'screenshot of training output' requirement.")

## Step 3: Evaluate Your Model

After training, this tests the model on images it has never seen — the test split from above. `model.eval()` turns off weight updates during this step.

Beyond the handout's exact-match accuracy, this also reports **Character Error Rate (CER)** and **Word Error Rate (WER)** — exact-match is strict (one wrong character anywhere in the line counts as fully wrong), while CER/WER are the standard OCR metrics and give a more graded sense of how close predictions are.

In [ ]:
import jiwer

# Use the best validation-CER checkpoint from training (Step 2 Extra: Checkpoint to Disk)
# if one was saved -- the model in memory right now is whatever epoch training happened to
# stop on, which isn't guaranteed to be the best one now that NUM_EPOCHS is more than a couple.
if os.path.isdir(BEST_MODEL_DIR) and os.listdir(BEST_MODEL_DIR):
    print(f"Loading best checkpoint by validation CER from {BEST_MODEL_DIR} for final evaluation.\n")
    model = VisionEncoderDecoderModel.from_pretrained(BEST_MODEL_DIR).to(device)
else:
    print("No best_model/ checkpoint found -- evaluating whatever model is currently in memory.\n")

model.eval()
print("=== Model Evaluation on Test Images ===\n")

results = []
with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].clone()
        labels[labels == -100] = processor.tokenizer.pad_token_id  # undo the training-time mask for decoding

        # num_beams=4: beam search instead of the handout's implicit greedy decoding
        # (num_beams=1). Greedy commits to the single best token at each step with no way
        # back; beam search keeps several candidate sequences alive and tends to produce
        # closer transcriptions for seq2seq generation tasks like this one.
        generated_ids = model.generate(pixel_values, max_length=MAX_LENGTH, num_beams=4)
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)
        actual_text = processor.batch_decode(labels, skip_special_tokens=True)

        for pred, actual in zip(generated_text, actual_text):
            pred, actual = pred.strip(), actual.strip()
            results.append({
                "predicted": pred,
                "actual": actual,
                "exact_match": pred == actual,
                "cer": jiwer.cer(actual, pred) if actual else None,
            })
            print(f"Predicted: {pred}")
            print(f"Actual: {actual}")
            print()

correct = sum(r["exact_match"] for r in results)
total = len(results)
accuracy = (correct / total) * 100 if total > 0 else 0

refs = [r["actual"] for r in results]
hyps = [r["predicted"] for r in results]
overall_cer = jiwer.cer(refs, hyps) if refs else float("nan")
overall_wer = jiwer.wer(refs, hyps) if refs else float("nan")

print(f"Accuracy: {accuracy:.1f}% ({correct}/{total} correct)")
print(f"Character Error Rate (CER): {overall_cer:.3f}")
print(f"Word Error Rate (WER): {overall_wer:.3f}")

### Step 3 Extra: Automatically Find the Worst Predictions

*(Not in the handout.)* The handout asks you to note 3-5 examples where the model got it wrong for Week 5. This pulls the worst ones by CER automatically instead of scrolling through the printed output above by hand.

In [ ]:
print("=== Examples where the model got it wrong (for your Week 5 discussion points) ===\n")

wrong = [r for r in results if not r["exact_match"]]
wrong_sorted = sorted(wrong, key=lambda r: (r["cer"] if r["cer"] is not None else 0), reverse=True)

if not wrong_sorted:
    print("No mistakes on the test set (or the test set is very small -- check the count above).")
else:
    for i, r in enumerate(wrong_sorted[:5], 1):
        print(f"{i}. Predicted: {r['predicted']}")
        print(f"   Actual:    {r['actual']}")
        print(f"   CER: {r['cer']:.3f}" if r["cer"] is not None else "   CER: n/a")
        print()

### Step 3 Extra: Visualize Sample Predictions

*(Not in the handout.)* A grid of actual test images next to what the model predicted vs. the ground truth — green border for an exact match, red for a mismatch.

In [ ]:
import textwrap
import random
from PIL import ImageDraw, ImageFont

# Map results back to their source image paths. test_loader has shuffle=False, so it
# iterates test_dataset in the same order as test_dataset.indices -- verified in testing.
test_image_paths = [test_dataset.dataset.data.iloc[i]["image"] for i in test_dataset.indices]
assert len(results) == len(test_image_paths), (
    f"Got {len(results)} predictions but {len(test_image_paths)} test images -- "
    "re-run the evaluation cell above before this one."
)
for r, img_path in zip(results, test_image_paths):
    r["image"] = img_path


import urllib.request

# The caption font (Urdu Nastaliq) isn't bundled with this notebook -- download it once
# so the fallback font_path below (used since no Week-3 FONTS dict is in memory here)
# actually points at a real file instead of a path that was never created.
os.makedirs("fonts", exist_ok=True)
NASTALIQ_FONT_PATH = "fonts/NotoNastaliqUrdu.ttf"
if not os.path.isfile(NASTALIQ_FONT_PATH):
    FONT_URL = (
        "https://raw.githubusercontent.com/google/fonts/main/"
        "ofl/notonastaliqurdu/NotoNastaliqUrdu%5Bwght%5D.ttf"
    )
    urllib.request.urlretrieve(FONT_URL, NASTALIQ_FONT_PATH)
    print(f"Downloaded Urdu Nastaliq font to {NASTALIQ_FONT_PATH}")


def show_prediction_grid(results, data_dir, n=9, cols=3, seed=0,
                          font_path=FONTS["nastaliq"] if "FONTS" in dir() else "fonts/NotoNastaliqUrdu.ttf",
                          thumb_size=230, caption_h=140, font_size=15):
    """Grid of sample test predictions: image + predicted vs. actual text.

    Captions are drawn with PIL using RTL shaping (direction='rtl', language='ur'),
    the same approach used for the Week 3 dataset-sample grid, since matplotlib
    cannot shape Arabic-script text on its own.
    """
    if not results:
        print("No results to show.")
        return
    random.seed(seed)
    sample = random.sample(results, k=min(n, len(results)))

    n_cols = min(cols, len(sample))
    n_rows = -(-len(sample) // n_cols)
    cell_w, cell_h = thumb_size, thumb_size + caption_h
    row_h = int(font_size * 2.1)

    grid_img = Image.new("RGB", (n_cols * cell_w, n_rows * cell_h), (255, 255, 255))
    font = ImageFont.truetype(font_path, font_size)
    label_font = ImageFont.load_default()

    for i, r in enumerate(sample):
        row_c, col_c = divmod(i, n_cols)
        thumb = Image.open(resolve_image_path(data_dir, r["image"])).convert("RGB")  # audit fix: same path-convention issue as the Dataset class
        thumb.thumbnail((thumb_size - 10, thumb_size - 10))
        cell = Image.new("RGB", (cell_w, cell_h), (255, 255, 255))
        cell.paste(thumb, ((cell_w - thumb.width) // 2, (thumb_size - thumb.height) // 2))

        draw = ImageDraw.Draw(cell)
        y = thumb_size + 8
        for label, text, color in [("Pred:", r["predicted"], (30, 60, 150)), ("True:", r["actual"], (20, 20, 20))]:
            draw.text((8, y), label, font=label_font, fill=(120, 120, 120))
            y += 14
            wrapped = textwrap.wrap(str(text), width=22)[:1]
            line = wrapped[0] if wrapped else str(text)
            if len(str(text)) > 22:
                line = line + "…"
            bbox = draw.textbbox((0, 0), line, font=font, direction="rtl", language="ur")
            tw = bbox[2] - bbox[0]
            draw.text((min(cell_w - 8, tw + 8), y), line, font=font, fill=color,
                       direction="rtl", language="ur", anchor="ra")
            y += row_h

        border_color = (85, 168, 104) if r["exact_match"] else (196, 78, 82)
        draw.rectangle([0, 0, cell_w - 1, cell_h - 1], outline=border_color, width=4)
        grid_img.paste(cell, (col_c * cell_w, row_c * cell_h))

    plt.figure(figsize=(n_cols * 3.2, n_rows * 4.1))
    plt.imshow(grid_img)
    plt.axis("off")
    n_correct = sum(r["exact_match"] for r in sample)
    plt.title(f"Sample Predictions ({n_correct}/{len(sample)} exact match) — green = correct, red = mismatch")
    plt.tight_layout()
    plt.show()

show_prediction_grid(results, DATA_DIR, n=9)

## Step 4: Save Your Model Locally

This saves the finished model to the cloned repo's data folder on the Colab VM's local disk, alongside the loss history and evaluation metrics. **This alone will NOT survive a Colab runtime disconnect or restart** — it's just staging before Step 5 pushes it somewhere durable.

In [ ]:
import json

# Saved next to the cloned data on the Colab VM's local disk. This is ephemeral --
# it disappears when the runtime recycles -- Step 5 pushes the real durable copy to the Hub.
SAVE_DIR = os.path.join(DATA_DIR, "model")
os.makedirs(SAVE_DIR, exist_ok=True)

try:
    model.save_pretrained(SAVE_DIR)
    processor.save_pretrained(SAVE_DIR)

    metrics = {
        "accuracy_pct": accuracy,
        "cer": overall_cer,
        "wer": overall_wer,
        "loss_first_batch": loss_history[0],
        "loss_last_batch": loss_history[-1],
        "epoch_avg_losses": epoch_avg_losses,
        "num_epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
    }
    with open(os.path.join(SAVE_DIR, "week4_metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    print(f"Model saved locally to: {SAVE_DIR}")
    print("This is on the Colab VM's disk only -- run Step 5 next to push it to the HF Hub")
    print("before this runtime disconnects, or you'll have to retrain.")
except OSError as e:
    raise RuntimeError(f"Couldn't save to {SAVE_DIR}: {e}. Check you have disk space and write permission.") from e

## Step 5: Push the Model to the Hugging Face Hub

*(Not in the handout.)* This is what actually makes the trained model durable past this Colab session. `push_to_hub()` uploads the model weights, processor config, and a generated model card to a repo on the Hub -- Week 5 (or anyone) can then reload it with `from_pretrained("<your-username>/<repo-name>")` instead of retraining from scratch.

**Fill in `HF_REPO_ID` below** (e.g. `your-hf-username/trocr-urdu-si26-week4`). If the repo doesn't exist yet, `push_to_hub` creates it automatically (as a public repo by default -- pass `private=True` in the `create_repo` call below if you'd rather it not be public).

In [ ]:
from huggingface_hub import HfApi, create_repo

# --- Fill this in ---
HF_REPO_ID = "hamnaheh/trocr-urdu-si26-week4"

create_repo(HF_REPO_ID, exist_ok=True, private=False)  # set private=True to keep it unlisted

commit_message = (
    f"Week 4 fine-tune: acc={accuracy:.1f}%, CER={overall_cer:.3f}, WER={overall_wer:.3f}, "
    f"{NUM_EPOCHS} epochs, batch_size={BATCH_SIZE}, lr={LEARNING_RATE}"
)

model.push_to_hub(HF_REPO_ID, commit_message=commit_message)
processor.push_to_hub(HF_REPO_ID, commit_message=commit_message)

# Also upload the metrics file saved in Step 4 so it lives alongside the model on the Hub.
api = HfApi()
api.upload_file(
    path_or_fileobj=os.path.join(SAVE_DIR, "week4_metrics.json"),
    path_in_repo="week4_metrics.json",
    repo_id=HF_REPO_ID,
    commit_message="Add Week 4 evaluation metrics",
)

print(f"Pushed to: https://huggingface.co/{HF_REPO_ID}")
print("Reload next week with:")
print(f'  model = VisionEncoderDecoderModel.from_pretrained("{HF_REPO_ID}")')
print(f'  processor = TrOCRProcessor.from_pretrained("{HF_REPO_ID}")')

After running Step 5, check the printed `huggingface.co/...` link to confirm the model repo exists and has files in it, before ending this Colab session.

## Saving This Notebook Back to GitHub

The model itself already lives on the Hugging Face Hub (Step 5) — don't try to `git push` it or the cloned data folder back to GitHub; both are large/binary and git handles that poorly (and the data folder is redundant with what's already in your repo anyway).

For the **notebook file only**:

1. In Colab, go to **File → Save a copy in GitHub**.
2. Pick your repo (`Urdu-OCR-Project-Code-Saviours-SI-26-Humna-Imran` or whichever you cloned in Step 0) and branch.
3. Optionally rename the file (e.g. `SI26_Week4_humna_colab.ipynb`) and add a commit message.
4. Click **OK** — this commits just the `.ipynb` (with its outputs) to your repo. It's a one-click action in the Colab UI, not something to script.

That's the artifact you submit as the "GitHub link to this notebook" in the checklist below.

After running this cell, check that the `model/` folder now exists under your data directory (`ls` it, or use the file explorer) before ending your Codespace session.

## Summary & Submission Checklist

- **GitHub link to this notebook** (with training + evaluation code) — push this file to your repo
- **`My model accuracy is X%`** — printed in Step 3 (`Accuracy: X% (n/m correct)`)
- **`Training loss went from X to X`** — printed at the end of the training loop cell
- **Screenshot of training output showing loss decreasing** — the Step 2 Extra chart (`training_loss.png`) covers this directly
- **3–5 wrong examples for Week 5** — auto-generated in Step 3 Extra

CER/WER, the prediction grid, and `week4_metrics.json` aren't required by the handout, but are there if useful context for Week 5's discussion.

One more thing worth flagging: the task text says to submit by Friday, July 25 — that's already past as of today (Sunday, July 26). Worth a quick check with your mentor on whether the deadline moved or that's just leftover boilerplate.